# Prototype Analysis on Sample

Dieses Notebook testet erste Analyseideen auf einem kleinen Sample der bereinigten Parking-Violations-Daten.

## Ziel

- processed Parquet-Daten aus HDFS laden
- kleines 1%-Sample erstellen
- Analysefragen testen
- häufigste Violation Codes untersuchen
- häufigste Vehicle Makes untersuchen
- zeitliche Muster nach Monat, Wochentag und Tageszeit prüfen
- beurteilen, welche Queries und Charts für die finale Analyse sinnvoll sind

Die finalen Analysen werden später in `src/4_Analysis` auf dem vollständigen Datensatz ausgeführt.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, desc

spark = SparkSession.builder \
    .appName("BDLC_Parking_Violations_Prototype") \
    .master("spark://bdlc-012.bdlc.ls.eee.intern:7077") \
    .config("spark.executor.cores", "4") \
    .config("spark.executor.memory", "15g") \
    .config("spark.cores.max", "12") \
    .getOrCreate()

spark


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/30 09:39:03 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
processed_path = "hdfs:///parking_violations/processed/parking_violations_cleaned_v5"

df = spark.read.parquet(processed_path)

df.groupBy("fy").count().orderBy("fy").show()

+----+--------+
|  fy|   count|
+----+--------+
|2022|  306279|
|2023|17246732|
|2024|16162180|
|2025|16251490|
|2026|     882|
+----+--------+



In [3]:
sample_df = df.sample(fraction=0.01, seed=42)

sample_count = sample_df.count()
sample_count

500258

In [4]:
sample_df.groupBy("violation_code", "violation_description_official") \
    .count() \
    .orderBy(desc("count")) \
    .show(20, truncate=False)


[Stage 7:======================================================>  (24 + 1) / 25]

+--------------+------------------------------+------+
|violation_code|violation_description_official|count |
+--------------+------------------------------+------+
|36            |PHTO SCHOOL ZN SPEED VIOLATION|169027|
|21            |NO PARKING-STREET CLEANING    |58415 |
|38            |FAIL TO DSPLY MUNI METER RECPT|35344 |
|14            |NO STANDING-DAY/TIME LIMITS   |24927 |
|5             |BUS LANE VIOLATION            |21377 |
|7             |FAILURE TO STOP AT RED LIGHT  |20600 |
|40            |FIRE HYDRANT                  |20216 |
|20            |NO PARKING-DAY/TIME LIMITS    |18899 |
|71            |INSP. STICKER-EXPIRED/MISSING |17683 |
|70            |REG. STICKER-EXPIRED/MISSING  |11924 |
|46            |DOUBLE PARKING                |10464 |
|37            |EXPIRED MUNI METER            |8064  |
|31            |NO STANDING-COMM METER ZONE   |7721  |
|74            |FRONT OR BACK PLATE MISSING   |7285  |
|69            |FAIL TO DISP. MUNI METER RECPT|7237  |
|19       

In [5]:
sample_df.groupBy("violation_code") \
    .count() \
    .orderBy(desc("count")) \
    .show(20, truncate=False)

[Stage 10:============================================>           (20 + 4) / 25]

+--------------+------+
|violation_code|count |
+--------------+------+
|36            |169027|
|21            |58415 |
|38            |35344 |
|14            |24927 |
|5             |21377 |
|7             |20600 |
|40            |20216 |
|20            |18899 |
|71            |17683 |
|70            |11924 |
|46            |10464 |
|37            |8064  |
|31            |7721  |
|74            |7285  |
|69            |7237  |
|19            |7219  |
|16            |6892  |
|12            |5418  |
|43            |5338  |
|15            |4892  |
+--------------+------+
only showing top 20 rows



In [6]:
sample_df.groupBy("vehicle_make") \
    .count() \
    .orderBy(desc("count")) \
    .show(20, truncate=False)

[Stage 13:============================================>           (20 + 4) / 25]

+------------+-----+
|vehicle_make|count|
+------------+-----+
|HONDA       |59042|
|TOYOT       |58842|
|FORD        |46871|
|NISSA       |39052|
|CHEVR       |26802|
|ME/BE       |25926|
|BMW         |24699|
|JEEP        |22766|
|HYUND       |17197|
|LEXUS       |12327|
|FRUEH       |11158|
|SUBAR       |11035|
|ACURA       |10843|
|KIA         |10264|
|DODGE       |9790 |
|AUDI        |9535 |
|MAZDA       |9331 |
|VOLKS       |9140 |
|INFIN       |7664 |
|RAM         |7464 |
+------------+-----+
only showing top 20 rows



In [7]:
sample_df.groupBy("fy", "issue_month") \
    .count() \
    .orderBy("fy", "issue_month") \
    .show(50)

[Stage 16:===================================>                    (16 + 4) / 25]

+----+-----------+-----+
|  fy|issue_month|count|
+----+-----------+-----+
|2022|          1|    8|
|2022|          2|    4|
|2022|          3|    1|
|2022|          4|    2|
|2022|          5|   11|
|2022|          6| 3088|
|2023|          1|13630|
|2023|          2|12850|
|2023|          3|15063|
|2023|          4|14017|
|2023|          5|15248|
|2023|          6|13456|
|2023|          7|13602|
|2023|          8|17648|
|2023|          9|15105|
|2023|         10|15384|
|2023|         11|14620|
|2023|         12|12756|
|2024|          1|12314|
|2024|          2|12396|
|2024|          3|13553|
|2024|          4|12835|
|2024|          5|14548|
|2024|          6|13953|
|2024|          7|15018|
|2024|          8|14738|
|2024|          9|12568|
|2024|         10|13892|
|2024|         11|13648|
|2024|         12|11698|
|2025|          1|12067|
|2025|          2|11892|
|2025|          3|14463|
|2025|          4|14924|
|2025|          5|14307|
|2025|          6|10461|
|2025|          7|14601|


26/05/30 09:39:21 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors
                                                                                

In [8]:
sample_df.groupBy("issue_weekday") \
    .count() \
    .orderBy("issue_weekday") \
    .show()

[Stage 19:========================================>               (18 + 4) / 25]

+-------------+-----+
|issue_weekday|count|
+-------------+-----+
|            1|48784|
|            2|72468|
|            3|80495|
|            4|75390|
|            5|81454|
|            6|78875|
|            7|62792|
+-------------+-----+



In [9]:
sample_df.filter(col("violation_hour").isNotNull()) \
    .groupBy("violation_hour") \
    .count() \
    .orderBy("violation_hour") \
    .show(24)

[Stage 22:======================================>                 (17 + 4) / 25]

+--------------+-----+
|violation_hour|count|
+--------------+-----+
|             0| 6402|
|             1| 7357|
|             2| 5683|
|             3| 4739|
|             4| 4410|
|             5| 6887|
|             6|15417|
|             7|27797|
|             8|42171|
|             9|44131|
|            10|35784|
|            11|44247|
|            12|40825|
|            13|38623|
|            14|35538|
|            15|30110|
|            16|24095|
|            17|20573|
|            18|15149|
|            19|11444|
|            20|11162|
|            21| 9987|
|            22| 8695|
|            23| 7925|
+--------------+-----+



## Erkenntnisse aus dem Prototyping

Das 1%-Sample enthält 500'468 Zeilen und ist damit gross genug, um Analyseideen zu testen.

Getestete Analysefragen:

1. **Welche Violation Codes kommen am häufigsten vor?**  
   `violation_code = 36` (PHTO SCHOOL ZN SPEED VIOLATION) ist im Sample mit Abstand am häufigsten. Für die finale Analyse wird `violation_description_official` verwendet, da diese eine eindeutige offizielle Beschreibung pro Code liefert.

2. **Welche Vehicle Makes erhalten am häufigsten Parking Violations?**  
   Im Sample sind `HONDA`, `TOYOT`, `FORD` und `NISSA` besonders häufig.

3. **Gibt es zeitliche Muster nach Monat?**  
   Die Monatsverteilung ist nach der Fiskaljahr-Korrektur (fy) über alle drei Jahre gleichmässig. FY2023 zeigt leicht erhöhte Werte in den Sommermonaten — vermutlich ein saisonales Muster.

4. **Gibt es zeitliche Muster nach Wochentag?**  
   Sonntag (Wochentag 1) weist deutlich weniger Verstösse auf als die Werktage.

5. **Gibt es zeitliche Muster nach Tageszeit?**  
   Im Sample zeigen sich hohe Werte insbesondere zwischen ca. 08:00 und 14:00 Uhr. Für diese Analyse werden nur Datensätze mit `violation_hour IS NOT NULL` verwendet.

Die getesteten Analysen werden im nächsten Schritt in `src/4_Analysis` auf dem vollständigen Datensatz ausgeführt.

In [10]:
spark.stop()